# Task B — MuRIL-large local OOF evaluation

This notebook evaluates MuRIL-large without relying on CodaBench. It trains one model for each of five stratified folds, predicts only the held-out fold, and computes macro-F1 and accuracy from the concatenated out-of-fold predictions.

| setting | value |
|---|---|
| encoder | `google/muril-large-cased` |
| TAPT | Task B training text + permitted OffensEval text, no holdout |
| classifier data | all 3,159 Task B rows, no deduplication |
| validation | 5-fold OOF; every row is predicted by a model that did not train on it |
| reinitialization | one final encoder layer |
| R-Drop | 0.5 |
| FGM | disabled for memory |
| sequence cap | 128 tokens |
| micro-batch | 2 with 8-step accumulation, effective batch 16 |
| classifier epochs | 6 |
| model seeds | one seed, `42` |

This is a local evaluation notebook, not a submission notebook. The official test labels are hidden, so the OOF score is the honest local estimate available while CodaBench submissions are closed. The first T4 attempt OOMed with a 192-token cap and batch 4; this version uses safer memory settings.

In [ ]:
import os, pathlib, shutil, subprocess, sys

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)
import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Build the MuRIL-large TAPT checkpoint

TAPT sees only Task B and permitted OffensEval text, not labels.

In [ ]:
TAPT_OUT = "artifacts/runs/tapt-muril-large-oof"
TAPT_LOG = "artifacts/logs/tapt_muril_large_oof.log"
if (pathlib.Path(TAPT_OUT) / "config.json").exists():
    print("using existing large OOF TAPT checkpoint:", TAPT_OUT)
else:
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--model", "google/muril-large-cased",
         "--corpus", "data/raw/multiclass_train.csv", "data/external/offenseval_kn.csv",
         "--val-frac", "0", "--min-words", "1", "--no-dedupe",
         "--epochs", "8", "--bs", "2", "--grad-accum", "8",
         "--eval-bs", "4", "--out", TAPT_OUT], log=TAPT_LOG)
assert (pathlib.Path(TAPT_OUT) / "config.json").exists(), "large TAPT checkpoint was not written"
print("large OOF TAPT checkpoint ready:", TAPT_OUT)


## 2. Train five held-out folds

Each fold trains on four-fifths of the 3,159 rows and evaluates on the remaining fifth. The micro-batch is 2, but gradient accumulation keeps the effective batch size at 16.

In [ ]:
TAG = "b_muril_large_oof"
TRAIN_LOG = pathlib.Path("artifacts/logs") / f"{TAG}.log"
RUN_DIR = pathlib.Path("artifacts/runs") / TAG
run([sys.executable, "-u", "-m", "hastika.task_b.train",
     "--tag", TAG, "--model", TAPT_OUT,
     "--folds", "5", "--no-dedupe", "--max-len", "128",
     "--reinit-layers", "1", "--rdrop", "0.5", "--no-fgm",
     "--lr", "1e-5", "--bs", "2", "--grad-accum", "8",
     "--eval-bs", "4", "--aux-weight", "0",
     "--seeds", "42", "--epochs", "6"], log=str(TRAIN_LOG))
assert (RUN_DIR / "oof_probs.npy").exists(), "OOF probabilities were not written"
print("MuRIL-large OOF training completed:", RUN_DIR)


## 3. Compute the local OOF score

Every score below is calculated from predictions made on rows excluded from that fold's training set.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score
LABELS = ["Gender", "Geo-political", "Others", "Political", "Religion", "Violence"]
train = pd.read_csv("data/raw/multiclass_train.csv")
y = train["Hate Category"].map(LABELS.index).to_numpy()
probs = np.load(RUN_DIR / "oof_probs.npy")
assert probs.shape == (len(train), len(LABELS)), f"unexpected OOF shape: {probs.shape}"
pred = probs.argmax(1)
macro_f1 = f1_score(y, pred, average="macro")
accuracy = accuracy_score(y, pred)
print({"macro_f1": round(macro_f1, 4), "accuracy": round(accuracy, 4)})
print(classification_report(y, pred, labels=range(len(LABELS)), target_names=LABELS, digits=4, zero_division=0))
pd.DataFrame([{"run": TAG, "macro_f1": macro_f1, "accuracy": accuracy}]).to_csv(RUN_DIR / "oof_summary.csv", index=False)


## 4. Preserve outputs

No submission ZIP is created. The OOF score is the purpose of this notebook; test probabilities are copied only as supplementary artifacts.

In [ ]:
OUT = pathlib.Path("/kaggle/working/muril_large_oof_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for name in ["oof_probs.npy", "test_probs.npy", "predictions.csv", "oof_summary.csv"]:
    source = RUN_DIR / name
    if source.exists():
        shutil.copy2(source, OUT / name)
for source in [TRAIN_LOG, pathlib.Path(TAPT_LOG)]:
    if source.exists():
        shutil.copy2(source, OUT / source.name)
print("download:", OUT)
print("files:", sorted(x.name for x in OUT.iterdir()))
